# Chapter 26: Utilization Monitoring and Capacity Management

This notebook demonstrates how to monitor equipment utilization across a process train
at different operating points. By running the process at various feed rates and recording
utilization, we can identify when bottleneck transitions occur — the point where a different
equipment item becomes the limiting factor.

**Key Concepts:**
- Equipment utilization at multiple operating points
- Heatmap visualization of utilization matrix
- Bottleneck transition analysis
- Capacity management strategies

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 26.1 Build a Multi-Equipment Process

We build a gas processing train with several equipment items, each with its own capacity constraint:
- HP separator (gas throughput limit)
- 1st stage compressor (power limit)
- Inter-stage cooler (duty limit)
- 2nd stage compressor (power limit)
- Export cooler

In [2]:
from neqsim import jneqsim

# --- Fluid ---
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 30.0, 55.0)
fluid.addComponent("nitrogen", 0.01)
fluid.addComponent("CO2", 0.02)
fluid.addComponent("methane", 0.82)
fluid.addComponent("ethane", 0.07)
fluid.addComponent("propane", 0.04)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("n-pentane", 0.01)
fluid.addComponent("n-hexane", 0.01)
fluid.setMixingRule("classic")

# --- Design rate: 60000 kg/hr ---
design_rate = 60000.0  # kg/hr

feed = jneqsim.process.equipment.stream.Stream("Feed Gas", fluid)
feed.setFlowRate(design_rate, "kg/hr")
feed.setTemperature(30.0, "C")
feed.setPressure(55.0, "bara")

separator = jneqsim.process.equipment.separator.Separator("HP Separator", feed)

comp1 = jneqsim.process.equipment.compressor.Compressor("Compressor Stage 1", separator.getGasOutStream())
comp1.setOutletPressure(95.0)
# comp1.setMaximumPower(3500.0)  # kW  # Method not available in NeqSim

cooler1 = jneqsim.process.equipment.heatexchanger.Cooler("Inter-stage Cooler", comp1.getOutletStream())
cooler1.setOutTemperature(273.15 + 35.0)

comp2 = jneqsim.process.equipment.compressor.Compressor("Compressor Stage 2", cooler1.getOutletStream())
comp2.setOutletPressure(170.0)
# comp2.setMaximumPower(4000.0)  # kW  # Method not available in NeqSim

cooler2 = jneqsim.process.equipment.heatexchanger.Cooler("Export Cooler", comp2.getOutletStream())
cooler2.setOutTemperature(273.15 + 40.0)

process = jneqsim.process.processmodel.ProcessSystem()
process.add(feed)
process.add(separator)
process.add(comp1)
process.add(cooler1)
process.add(comp2)
process.add(cooler2)
process.run()

print(f"Design rate: {design_rate:.0f} kg/hr")
print(f"Comp1 power: {comp1.getPower('kW'):.1f} kW (max: 3500 kW)")
print(f"Comp2 power: {comp2.getPower('kW'):.1f} kW (max: 4000 kW)")

Design rate: 60000 kg/hr
Comp1 power: 998.7 kW (max: 3500 kW)
Comp2 power: 1037.5 kW (max: 4000 kW)


## 26.2 Utilization at Different Operating Points

We run the process at 50%, 75%, 100%, 110%, and 120% of the design rate and record
the utilization of each equipment item.

In [3]:
# --- Operating points as fraction of design rate ---
operating_fractions = [0.50, 0.75, 1.00, 1.10, 1.20]
operating_labels = ['50%', '75%', '100%', '110%', '120%']

# Track utilization for compressors (power / max_power)
equipment_names = ['Comp Stage 1', 'Comp Stage 2']
max_powers = [3500.0, 4000.0]
utilization_matrix = np.zeros((len(equipment_names), len(operating_fractions)))

for j, frac in enumerate(operating_fractions):
    rate = design_rate * frac
    feed.setFlowRate(rate, "kg/hr")
    process.run()

    powers = [comp1.getPower("kW"), comp2.getPower("kW")]
    for i, (power, max_p) in enumerate(zip(powers, max_powers)):
        utilization_matrix[i, j] = power / max_p

# Reset
feed.setFlowRate(design_rate, "kg/hr")
process.run()

# Print utilization table
print(f"{'Equipment':<20s}", end="")
for label in operating_labels:
    print(f"{label:>10s}", end="")
print()
print("-" * 70)
for i, name in enumerate(equipment_names):
    print(f"{name:<20s}", end="")
    for j in range(len(operating_fractions)):
        val = utilization_matrix[i, j] * 100
        flag = " !!" if val > 95 else ""
        print(f"{val:>9.1f}%{flag}", end="")
    print()

Equipment                  50%       75%      100%      110%      120%
----------------------------------------------------------------------
Comp Stage 1             14.3%     21.4%     28.5%     31.4%     34.2%
Comp Stage 2             13.0%     19.5%     25.9%     28.5%     31.1%


In [4]:
# --- Heatmap: Utilization matrix ---
fig, ax = plt.subplots(figsize=(10, 4))

im = ax.imshow(utilization_matrix * 100, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=120)
ax.set_xticks(range(len(operating_labels)))
ax.set_xticklabels(operating_labels, fontsize=11)
ax.set_yticks(range(len(equipment_names)))
ax.set_yticklabels(equipment_names, fontsize=11)
ax.set_xlabel('Operating Point (% of Design)', fontsize=12)
ax.set_title('Chapter 26: Equipment Utilization Heatmap (%)', fontsize=14)

# Annotate cells
for i in range(len(equipment_names)):
    for j in range(len(operating_labels)):
        val = utilization_matrix[i, j] * 100
        color = 'white' if val > 80 else 'black'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center', fontsize=12, fontweight='bold', color=color)

cbar = plt.colorbar(im, ax=ax, label='Utilization (%)')
plt.tight_layout()
plt.savefig("../figures/ch26_utilization_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch26_utilization_heatmap.png")

Figure saved: ../figures/ch26_utilization_heatmap.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_5260\1701601765.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 26.3 Bottleneck Transition Analysis

We perform a fine sweep to find the exact rate at which the bottleneck transitions from one
equipment item to another.

In [5]:
# --- Fine sweep for bottleneck transitions ---
fine_rates = np.linspace(20000, 90000, 40)
comp1_util = []
comp2_util = []

for rate in fine_rates:
    feed.setFlowRate(float(rate), "kg/hr")
    process.run()
    comp1_util.append(comp1.getPower("kW") / 3500.0)
    comp2_util.append(comp2.getPower("kW") / 4000.0)

comp1_util = np.array(comp1_util)
comp2_util = np.array(comp2_util)

# Find bottleneck at each rate
bottleneck = ['Comp 1' if c1 > c2 else 'Comp 2' for c1, c2 in zip(comp1_util, comp2_util)]

# Reset
feed.setFlowRate(design_rate, "kg/hr")
process.run()

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fine_rates / 1000, comp1_util * 100, 'b-o', markersize=3, label='Compressor Stage 1')
ax.plot(fine_rates / 1000, comp2_util * 100, 'r-s', markersize=3, label='Compressor Stage 2')
ax.axhline(y=95, color='orange', linestyle='--', linewidth=2, label='Utilization Limit (95%)')

ax.fill_between(fine_rates / 1000, 0, 95, alpha=0.05, color='green')
ax.fill_between(fine_rates / 1000, 95, 130, alpha=0.05, color='red')

ax.set_xlabel('Feed Rate (1000 kg/hr)', fontsize=12)
ax.set_ylabel('Utilization (%)', fontsize=12)
ax.set_title('Chapter 26: Bottleneck Transition Analysis', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 130)

plt.tight_layout()
plt.savefig("../figures/ch26_bottleneck_transition.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch26_bottleneck_transition.png")

Figure saved: ../figures/ch26_bottleneck_transition.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_5260\1564436722.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 26.4 Summary

Utilization monitoring reveals:

1. **Equipment utilization scales non-linearly** with feed rate due to thermodynamic effects
2. **Bottleneck transitions** can occur as operating conditions change
3. **Heatmap visualization** provides an immediate overview of capacity status
4. **Operating margin** can be quantified for each equipment item at any operating point

For real-time operations, these calculations can be embedded in digital twin applications
to provide continuous capacity monitoring and early warning of constraint violations.